## Setup

`.env` must contain the following values:
```
SERVER_IP=your_server_ip
FTP_PORT=21
FTP_USER=your_username
FTP_PASSWORD=your_password
```

Use the same `.venv` you use for dev here with python >= 3.12.10

## Getting matchids from db

In [ ]:
from pathlib import Path
import sqlite3


# --- PATHS ---


BASE_DIR = Path().resolve().parents[2] / "Internomat"
DB_FILE = BASE_DIR / "internomat.db"
DEMO_DIR = BASE_DIR / "test_demos"

print("Using DB:", DB_FILE)


# --- DB ---


def get_conn_local():
    conn = sqlite3.connect(DB_FILE, timeout=10)
    conn.row_factory = sqlite3.Row
    return conn

def get_all_matches_with_maps_local():
    with get_conn_local() as conn:
        rows = conn.execute("""
        SELECT 
            m.match_id,
            m.team1_name,
            m.team2_name,
            mm.map_number,
            mm.map_name
        FROM matches m
        LEFT JOIN match_maps mm 
            ON m.match_id = mm.match_id
        ORDER BY m.match_id, mm.map_number
        """).fetchall()

    matches = {}

    for row in rows:
        match_id = int(row["match_id"])

        if match_id not in matches:
            matches[match_id] = {
                "match_id": match_id,
                "team1": row["team1_name"],
                "team2": row["team2_name"],
                "maps": []
            }

        if row["map_number"] is not None:
            matches[match_id]["maps"].append({
                "map_number": int(row["map_number"]),
                "map_name": row["map_name"]
            })

    return list(matches.values())


# --- SHARED UTILS ---


def extract_match_id(filename):
    try:
        return int(filename.split("_")[2])
    except:
        return None

def build_match_id_set(matches):
    return set(m["match_id"] for m in matches)


# --- LOAD MATCHES ---


matches = get_all_matches_with_maps_local()
valid_match_ids = build_match_id_set(matches)

print(f"\nLoaded {len(matches)} matches")
print(f"Valid match_ids: {sorted(valid_match_ids)}")


## Getting all .dem from server (for now)

In [ ]:
from ftplib import FTP
from dotenv import load_dotenv
import os
from tqdm import tqdm

# --- ENV ---

env_path = BASE_DIR / ".env"
load_dotenv(env_path)

FTP_HOST = os.getenv("SERVER_IP")
FTP_PORT = int(os.getenv("FTP_PORT", 21))
FTP_USER = os.getenv("FTP_USER")
FTP_PASSWORD = os.getenv("FTP_PASSWORD")

assert FTP_HOST and FTP_USER and FTP_PASSWORD, "Missing FTP env vars"


# --- HELPERS ---

def extract_parts(filename):
    # 2026-03-20_22-48-10_4_cs_office_team_...vs_...
    parts = filename.replace(".dem", "").split("_")

    try:
        date = parts[0]
        time = parts[1]
        match_id = int(parts[2])
        map_name = f"{parts[3]}_{parts[4]}"
        return date, time, match_id, map_name
    except:
        return None, None, None, None


def get_map_number(match, map_name):
    for m in match["maps"]:
        if m["map_name"] == map_name:
            return m["map_number"]
    return None


def build_match_lookup(matches):
    return {m["match_id"]: m for m in matches}


# --- FTP CONFIG ---

REMOTE_DIR = "/cs2/game/csgo/MatchZy"
DEMO_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n[FTP] Connecting to {FTP_HOST}:{FTP_PORT}...")

ftp = FTP()

try:
    ftp.connect(FTP_HOST, FTP_PORT, timeout=10)
    ftp.login(FTP_USER, FTP_PASSWORD)
    ftp.set_pasv(True)

    print("[FTP] Connected")

    ftp.cwd(REMOTE_DIR)
    files = ftp.nlst()

    downloaded, skipped, ignored = 0, 0, 0

    match_lookup = build_match_lookup(matches)

    print(f"[FTP] Found {len(files)} files")

    for file in files:

        if not file.endswith(".dem"):
            continue

        date, time, match_id, map_name = extract_parts(file)

        if match_id is None or match_id <= 0:
            ignored += 1
            continue

        if match_id not in valid_match_ids:
            ignored += 1
            continue

        match = match_lookup[match_id]
        map_number = get_map_number(match, map_name)



        # --- NEW NORMALIZED NAME ---
        normalized_name = f"{date}_{time}_match_{match_id}_map_{map_number}_{map_name}.dem"
        local_file = DEMO_DIR / normalized_name

        if local_file.exists():
            print(f"[SKIP] {normalized_name}")
            skipped += 1
            continue

        print(f"[DOWNLOAD] {file}")
        print(f"           → {normalized_name}")

        try:
            filesize = ftp.size(file)

            with open(local_file, "wb") as f, tqdm(
                total=filesize,
                unit="B",
                unit_scale=True,
                desc=normalized_name,
                leave=False
            ) as pbar:

                def callback(data):
                    f.write(data)
                    pbar.update(len(data))

                ftp.retrbinary(f"RETR {file}", callback)

            downloaded += 1

        except Exception as e:
            print(f"[ERROR] {file}: {e}")

    print("\n[FTP] Done")
    print(f"Downloaded: {downloaded}")
    print(f"Skipped: {skipped}")
    print(f"Ignored: {ignored}")

finally:
    try:
        ftp.quit()
    except:
        pass

## Match demo files to local matches

In [ ]:
from pathlib import Path
from awpy.demo import Demo


DEMO_DIR = Path().resolve().parents[2] / "Internomat" / "test_demos"


# --- HELPERS ---

def extract_ids_from_normalized(filename):
    # 2026-03-20_22-48-10_match_4_map_1_cs_office.dem
    parts = filename.replace(".dem", "").split("_")

    try:
        # find dynamic positions (safer than fixed indices)
        match_idx = parts.index("match")
        map_idx = parts.index("map")

        match_id = int(parts[match_idx + 1])
        map_number = int(parts[map_idx + 1])

        return match_id, map_number

    except Exception:
        return None, None


# --- MATCH + LOAD DEMOS ---

def match_and_load_demos():
    demo_files = list(DEMO_DIR.glob("*.dem"))

    print(f"\n[Matcher] Found {len(demo_files)} normalized demos\n")

    results = []
    failed = []

    for file in demo_files:
        match_id, map_number = extract_ids_from_normalized(file.name)

        if match_id is None or map_number is None:
            print(f"[SKIP] Invalid normalized file: {file.name}")
            continue

        print(f"[LOAD] {file.name} (match={match_id}, map={map_number})")

        try:
            demo = Demo(str(file))
            demo.parse()

            results.append({
                "file": file,
                "match_id": match_id,
                "map_number": map_number,
                "demo": demo
            })

        except Exception as e:
            print(f"[FAILED] {file.name}: {e}")
            failed.append(file.name)

    print(f"\n[Matcher] Loaded {len(results)} demos successfully")
    print(f"[Matcher] Failed: {len(failed)}")

    if failed:
        print("\nFailed demos:")
        for f in failed:
            print(f)

    return results


# --- RUN ---

matched = match_and_load_demos()

In [ ]:
# --- TEST PARSE SINGLE DEMO ---
def parse_demo_filename(filename):
    demo = Demo(str(filename))
    demo.parse()

filename = "2026-03-20_22-48-10_match_4_map_1_cs_office.dem"
#parse_demo_filename(DEMO_DIR / filename)

## Parser

In [ ]:
import pandas as pd
import polars as pl



# --- PARSER ---


def parse_demo_full(demo):
    data = {}

    data["header"] = demo.header

    table_map = {
        "rounds": "rounds",
        "kills": "kills",
        "damages": "damages",
        "grenades": "grenades",
        "shots": "shots",
        "footsteps": "footsteps",
        "smokes": "smokes",
        "infernos": "infernos",
        "bomb": "bomb",
        "ticks": "ticks",
        "player_round_totals": "player_round_totals",
        "server_cvars": "server_cvars",
    }

    for key, attr in table_map.items():
        try:
            value = getattr(demo, attr, None)

            if value is None:
                data[key] = None
                continue

            if isinstance(value, pd.DataFrame):
                df = value
            elif isinstance(value, list):
                df = pd.DataFrame(value)
            else:
                df = value

            data[key] = df

        except Exception:
            data[key] = None

    return data



# --- HELPERS ---


def print_headers(data, preview=False):
    print("\n=== TABLE HEADERS ===\n")

    for key, value in data.items():
        print(f"[TABLE] {key}")

        if value is None:
            print("  -> None\n")
            continue

        if isinstance(value, pd.DataFrame):
            print(f"  Rows: {len(value)}")
            print(f"  Columns ({len(value.columns)}):")
            for col in value.columns:
                print(f"    - {col}")

            if preview:
                print("\n  Preview:")
                print(value.head(3))

        elif isinstance(value, pl.DataFrame):
            print(f"  Rows: {value.height}")
            print(f"  Columns ({len(value.columns)}):")
            for col in value.columns:
                print(f"    - {col}")

        else:
            print(f"  Type: {type(value).__name__}")

        print("")


def get_cvars(data):
    cvars = data.get("server_cvars")
    if cvars is None:
        print("[WARN] No CVARs")
    return cvars



# --- BUILD DEMO DICT ---


def build_demo_data(matched):
    demo_dict = {}
    failed = 0

    print("\n[Parser] Building structured datasets...\n")

    for entry in matched:
        match_id = entry["match_id"]
        map_number = entry["map_number"]
        demo = entry["demo"]
        filename = entry["file"].name

        key = (match_id, map_number)

        print(f"[PARSE] {filename} → match={match_id}, map={map_number}")

        try:
            data = parse_demo_full(demo)

            demo_dict[key] = data

        except Exception as e:
            print(f"[FAILED] {filename}: {e}")
            failed += 1

    print("\n[Parser] Done")
    print(f"Parsed: {len(demo_dict)}")
    print(f"Failed: {failed}")

    return demo_dict



# --- RUN ---


demo_data = build_demo_data(matched)



# --- OPTIONAL INSPECTION ---


# pick first dataset to inspect
if demo_data:
    sample_key = list(demo_data.keys())[0]
    print(f"\n[Inspect] match={sample_key[0]} map={sample_key[1]}")

    sample_data = demo_data[sample_key]

    print_headers(sample_data)
    # print header table
    print("\n=== HEADER ===\n")
    for k, v in sample_data["header"].items():
        print(f"{k}: {v}")

## Analytics

In [ ]:
import polars as pl


def build_player_stats_single(data):
    # --- SOURCE TABLES ---
    kills = data["kills"]
    damages = data["damages"]
    shots = data["shots"]
    rounds = data["rounds"]
    players_df = data["player_round_totals"]

    if kills is None or damages is None or shots is None or players_df is None:
        return None

    # convert to polars if needed
    if not isinstance(kills, pl.DataFrame):
        kills = pl.from_pandas(kills)
    if not isinstance(damages, pl.DataFrame):
        damages = pl.from_pandas(damages)
    if not isinstance(shots, pl.DataFrame):
        shots = pl.from_pandas(shots)
    if not isinstance(players_df, pl.DataFrame):
        players_df = pl.from_pandas(players_df)
    if not isinstance(rounds, pl.DataFrame):
        rounds = pl.from_pandas(rounds)

    # --- PLAYER INDEX ---
    players = (
        players_df
        .filter(pl.col("side") == "all")
        .select(["steamid", "name"])
        .unique(subset=["steamid"])
    )

    # --- NORMALIZE EVENTS ---
    kills_norm = kills.select([
        pl.col("attacker_steamid").alias("steamid"),
        pl.lit(1).alias("kills"),
        pl.lit(0).alias("deaths"),
        pl.lit(0).alias("damage"),
        pl.lit(0).alias("shots"),
    ])

    deaths_norm = kills.select([
        pl.col("victim_steamid").alias("steamid"),
        pl.lit(0).alias("kills"),
        pl.lit(1).alias("deaths"),
        pl.lit(0).alias("damage"),
        pl.lit(0).alias("shots"),
    ])

    damage_norm = damages.select([
        pl.col("attacker_steamid").alias("steamid"),
        pl.lit(0).alias("kills"),
        pl.lit(0).alias("deaths"),
        pl.col("dmg_health_real").alias("damage"),
        pl.lit(0).alias("shots"),
    ])

    shots_norm = shots.select([
        pl.col("player_steamid").alias("steamid"),
        pl.lit(0).alias("kills"),
        pl.lit(0).alias("deaths"),
        pl.lit(0).alias("damage"),
        pl.lit(1).alias("shots"),
    ])

    all_events = pl.concat([
        kills_norm,
        deaths_norm,
        damage_norm,
        shots_norm
    ])

    # --- AGG ---
    stats = (
        all_events
        .group_by("steamid")
        .agg([
            pl.sum("kills"),
            pl.sum("deaths"),
            pl.sum("damage"),
            pl.sum("shots"),
        ])
    )

    # --- DERIVED ---
    n_rounds = max(1, rounds.height)

    stats = stats.with_columns([
        (pl.col("kills") / pl.when(pl.col("deaths") == 0).then(1).otherwise(pl.col("deaths"))).alias("kd"),
        (pl.col("damage") / n_rounds).alias("adr"),
        (pl.col("kills") / pl.when(pl.col("shots") == 0).then(1).otherwise(pl.col("shots"))).alias("accuracy")
    ])

    stats = stats.join(players, on="steamid", how="left")

    return stats



# --- MULTI DEMO WRAPPER ---


def build_player_stats_all(demo_data):
    print("[Stats] Building across all demos...\n")

    per_demo_stats = []

    for (match_id, map_number), data in demo_data.items():
        print(f"[Stats] match={match_id} map={map_number}")

        stats = build_player_stats_single(data)

        if stats is None:
            print("  -> skipped (missing data)")
            continue

        stats = stats.with_columns([
            pl.lit(match_id).alias("match_id"),
            pl.lit(map_number).alias("map_number")
        ])

        per_demo_stats.append(stats)

    if not per_demo_stats:
        print("[Stats] No valid demos")
        return None

    # --- CONCAT ALL ---
    all_stats = pl.concat(per_demo_stats)

    # --- GLOBAL AGG ---
    global_stats = (
        all_stats
        .group_by("steamid")
        .agg([
            pl.first("name"),
            pl.sum("kills"),
            pl.sum("deaths"),
            pl.sum("damage"),
            pl.sum("shots"),
        ])
    )

    global_stats = global_stats.with_columns([
        (pl.col("kills") / pl.when(pl.col("deaths") == 0).then(1).otherwise(pl.col("deaths"))).alias("kd"),
        (pl.col("damage") / pl.count()).alias("adr"),  # avg across demos
        (pl.col("kills") / pl.when(pl.col("shots") == 0).then(1).otherwise(pl.col("shots"))).alias("accuracy")
    ])

    global_stats = global_stats.sort("kills", descending=True)

    print("\n[Stats] Done\n")

    display(global_stats)

    return global_stats, all_stats



# --- RUN ---


player_stats_global, player_stats_per_demo = build_player_stats_all(demo_data)

## Filter

In [ ]:
import polars as pl


# ==============================
# --- SINGLE DEMO ---
# ==============================

def build_round_windows_single(data):
    rounds = data.get("rounds")

    if rounds is None:
        return None

    if not isinstance(rounds, pl.DataFrame):
        rounds = pl.from_pandas(rounds)

    round_windows = (
        rounds
        .select([
            "round_num",
            pl.col("freeze_end").alias("live_start"),
            pl.col("end").alias("live_end"),
        ])
        .sort("round_num")
    )

    return round_windows


# ==============================
# --- MULTI DEMO ---
# ==============================

def build_round_windows_all(demo_data):
    print("[Windows] Building round windows across all demos...\n")

    results = {}

    for (match_id, map_number), data in demo_data.items():
        print(f"[Windows] match={match_id} map={map_number}")

        windows = build_round_windows_single(data)

        if windows is None:
            print("  -> skipped (no rounds)")
            continue

        results[(match_id, map_number)] = windows

    print("\n[Windows] Done")

    # optional preview
    if results:
        first_key = list(results.keys())[0]
        print(f"\n[Preview] match={first_key[0]} map={first_key[1]}")
        display(results[first_key])

    return results


# ==============================
# --- RUN ---
# ==============================

round_windows_all = build_round_windows_all(demo_data)

In [ ]:
import polars as pl


# extract tickrate from demo
def get_tickrate(data):
    header = data.get("header")

    if header is None:
        raise ValueError("Missing demo header")

    # awpy usually provides this
    tickrate = header.get("tickrate")

    if tickrate is None:
        # fallback (common CS2 default)
        tickrate = 128

    return float(tickrate)


# single demo
def build_movement_stats_single(data, live_round_windows):
    print("[Movement] Building...")

    tickrate = get_tickrate(data)
    print(f"[Movement] Tickrate: {tickrate}")

    ticks = data["ticks"]
    players_df = data["player_round_totals"]

    # players
    players = (
        players_df
        .filter(pl.col("side") == "all")
        .select(["steamid", "name"])
        .unique(subset=["steamid"])
    )

    # join windows
    ticks = ticks.join(
        live_round_windows,
        on="round_num",
        how="left"
    )

    # sort
    ticks = ticks.sort(["steamid", "tick"])

    # deltas
    ticks = ticks.with_columns([
        pl.col("X").diff().over("steamid").alias("dx"),
        pl.col("Y").diff().over("steamid").alias("dy"),
        pl.col("Z").diff().over("steamid").alias("dz"),
        pl.col("tick").diff().over("steamid").alias("dt"),
    ])

    # distance
    ticks = ticks.with_columns([
        ((pl.col("dx")**2 + pl.col("dy")**2 + pl.col("dz")**2).sqrt()).alias("distance")
    ])

    # speed
    ticks = ticks.with_columns([
        (pl.col("distance") / pl.col("dt").fill_null(1)).alias("units_per_tick")
    ])

    ticks = ticks.with_columns([
        (pl.col("units_per_tick") * tickrate).alias("speed")
    ])

    # live window
    ticks = ticks.with_columns([
        (
            (pl.col("tick") >= pl.col("live_start")) &
            (pl.col("tick") <= pl.col("live_end"))
        ).alias("in_live")
    ])

    # filtered metrics
    ticks = ticks.with_columns([
        pl.when((pl.col("health") > 0) & pl.col("in_live"))
        .then(pl.col("distance"))
        .otherwise(0)
        .alias("distance_alive"),

        pl.when(
            (pl.col("health") > 0) &
            pl.col("in_live") &
            (pl.col("speed") < 400)
        )
        .then(pl.col("speed"))
        .otherwise(None)
        .alias("speed_valid"),

        pl.when((pl.col("health") > 0) & pl.col("in_live"))
        .then(1)
        .otherwise(0)
        .alias("alive_tick")
    ])

    # aggregate
    movement = (
        ticks
        .group_by("steamid")
        .agg([
            pl.sum("distance_alive").alias("total_distance_units"),
            pl.max("speed_valid").alias("max_speed_units_s"),
            pl.sum("alive_tick").alias("ticks_alive")
        ])
    )

    n_rounds = live_round_windows.height

    movement = movement.with_columns([
        (pl.col("ticks_alive") / tickrate).alias("alive_seconds"),

        (
            pl.col("total_distance_units") /
            (pl.col("ticks_alive") / tickrate)
        ).alias("avg_speed_units_s"),

        (pl.col("total_distance_units") / n_rounds).alias("distance_per_round_units"),

        (pl.col("total_distance_units") * 0.0254).alias("total_distance_m"),

        (
            (pl.col("total_distance_units") /
             (pl.col("ticks_alive") / tickrate)) * 0.0254
        ).alias("avg_speed_m_s"),
    ])

    movement = movement.join(players, on="steamid", how="left")

    movement = movement.select([
        "steamid",
        "name",
        "total_distance_units",
        "total_distance_m",
        "distance_per_round_units",
        "avg_speed_units_s",
        "avg_speed_m_s",
        "max_speed_units_s",
        "ticks_alive",
        "alive_seconds",
    ])

    movement = movement.fill_null(0)
    movement = movement.sort("total_distance_units", descending=True)

    print("[Movement] Done")

    return movement, ticks, players


# multi demo
def build_movement_stats_all(demo_data, round_windows_all):
    print("[Movement] Processing all demos...\n")

    results = {}

    for key in demo_data:
        data = demo_data[key]
        windows = round_windows_all.get(key)

        if windows is None:
            continue

        print(f"[Movement] match={key[0]} map={key[1]}")

        movement, ticks, players = build_movement_stats_single(data, windows)

        results[key] = {
            "movement": movement,
            "ticks": ticks,
            "players": players
        }

    print("\n[Movement] Done")

    return results


# run
movement_data = build_movement_stats_all(demo_data, round_windows_all)


In [ ]:
import matplotlib.pyplot as plt
import math
import polars as pl
import matplotlib.cm as cm

BIN_TICKS = 32
BIN_SIZE = None  # will be set dynamically


def plot_speed_per_round(data, ticks, players, live_round_windows, player_filter=None):
    print("[Plot] Building ROUND-SPLIT curves...")

    tickrate = get_tickrate(data)
    print(f"[Plot] Tickrate: {tickrate}")

    global BIN_SIZE
    BIN_SIZE = BIN_TICKS / tickrate

    # filter players
    if player_filter is not None:
        if not isinstance(player_filter, list):
            player_filter = [player_filter]

        if all(isinstance(p, str) for p in player_filter):
            players = players.filter(pl.col("name").is_in(player_filter))
        else:
            players = players.filter(pl.col("steamid").is_in(player_filter))

        selected_ids = players["steamid"].to_list()
        ticks = ticks.filter(pl.col("steamid").is_in(selected_ids))

        print(f"[Plot] Filtering: {players['name'].to_list()}")

    # join windows
    ticks_plot = ticks.join(
        live_round_windows,
        on="round_num",
        how="left"
    )

    # round time
    ticks_plot = ticks_plot.with_columns([
        ((pl.col("tick") - pl.col("live_start")) / tickrate).alias("round_time_sec")
    ])

    # live filter
    ticks_plot = ticks_plot.with_columns([
        (
            (pl.col("tick") >= pl.col("live_start")) &
            (pl.col("tick") <= pl.col("live_end"))
        ).alias("in_live")
    ]).filter(pl.col("in_live"))

    # speed
    ticks_plot = ticks_plot.with_columns([
        pl.when(
            (pl.col("dt") <= 2) &
            (pl.col("health") > 0)
        )
        .then(pl.col("speed") * 0.0254)
        .otherwise(None)
        .alias("speed_m_s")
    ])

    # player map
    player_dicts = players.to_dicts()
    player_map = {p["steamid"]: p["name"] for p in player_dicts}

    # colors
    cmap = cm.get_cmap("tab10", len(player_dicts))
    color_map = {
        p["steamid"]: cmap(i)
        for i, p in enumerate(player_dicts)
    }

    # mode
    use_raw = players.height == 0

    if use_raw:
        print("[Plot] RAW")
        agg = (
            ticks_plot
            .select(["round_num", "steamid", "round_time_sec", "speed_m_s"])
            .sort(["round_num", "steamid", "round_time_sec"])
        )
    else:
        print("[Plot] SMOOTHED")

        ticks_plot = ticks_plot.with_columns([
            (pl.col("round_time_sec") // BIN_SIZE).alias("time_bin")
        ])

        agg = (
            ticks_plot
            .group_by(["round_num", "steamid", "time_bin"])
            .agg(pl.median("speed_m_s").alias("speed_m_s"))
            .sort(["round_num", "steamid", "time_bin"])
        )

    # layout
    rounds = agg["round_num"].unique().to_list()
    cols = 4
    rows = math.ceil(len(rounds) / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 6, rows * 3), dpi=200)
    axes = axes.flatten()

    # loop rounds
    for i, r in enumerate(rounds):
        ax = axes[i]
        df_r = agg.filter(pl.col("round_num") == r)

        # stats (shared)
        stats = (
            ticks_plot
            .filter(
                (pl.col("round_num") == r) &
                (pl.col("health") > 0) &
                (pl.col("dt") <= 2)
            )
            .group_by("steamid")
            .agg([
                pl.mean("speed_m_s").alias("avg_speed"),
                (pl.count() / tickrate).alias("time_alive_sec")
            ])
        )

        # plot lines
        for p in player_dicts:
            sid = p["steamid"]
            name = p["name"]

            df_p = df_r.filter(pl.col("steamid") == sid)
            if df_p.height == 0:
                continue

            if use_raw:
                time_sec = df_p["round_time_sec"].to_list()
            else:
                time_sec = (df_p["time_bin"] * BIN_SIZE).to_list()

            speed = df_p["speed_m_s"].to_list()

            ax.plot(
                time_sec,
                speed,
                linewidth=1.2,
                alpha=0.9,
                color=color_map[sid],
                label=name
            )

        # labels / info
        if stats.height > 0:
            if players.height == 1:
                row = stats.to_dicts()[0]
                name = player_map.get(row["steamid"], "?")

                ax.text(
                    0.02, 0.02,
                    f"{name}\n{row['avg_speed']:.1f} m/s\n{row['time_alive_sec']:.1f}s alive",
                    transform=ax.transAxes,
                    fontsize=9,
                    va="bottom",
                    bbox=dict(boxstyle="round", alpha=0.2)
                )
            else:
                lines = []
                for row in stats.to_dicts():
                    name = player_map.get(row["steamid"], "?")
                    lines.append(
                        f"{name}: {row['avg_speed']:.1f} m/s | {row['time_alive_sec']:.1f}s"
                    )

                ax.text(
                    0.98, 0.98,
                    "\n".join(lines),
                    transform=ax.transAxes,
                    fontsize=7,
                    va="top",
                    ha="right",
                    bbox=dict(boxstyle="round", alpha=0.2)
                )

                ax.legend(fontsize=7, loc="lower left")

        # axis
        max_time = (
            ticks_plot
            .filter(pl.col("round_num") == r)
            .select(pl.col("round_time_sec").max())
            .item()
        )

        ax.set_xlim(0, min(120, max_time + 1))
        ax.set_ylim(0, 15)

        ax.set_title(f"Round {r}", fontsize=10)
        ax.set_xlabel("Time (s)", fontsize=8)
        ax.set_ylabel("Speed", fontsize=8)
        ax.grid(alpha=0.2)

        # shift line
        SHIFT_SPEED = 5.95
        ax.axhline(y=SHIFT_SPEED, linestyle="--", linewidth=1.0, alpha=0.6, color="black")

    # cleanup
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    # --- GLOBAL TITLE ---

    header = data.get("header", {})
    map_name = header.get("map_name", "unknown")

    n_rounds = len(rounds)

    # player list
    player_names = players["name"].to_list()
    player_names_sorted = sorted(player_names)

    players_text = ", ".join(player_names_sorted)

    title_text = f"{map_name} | {n_rounds} rounds"
    subtitle_text = f"Players: {players_text}"

    fig.suptitle(
        title_text,
        fontsize=16,
        y=1.04
    )

    fig.text(
        0.5, 1.01,
        subtitle_text,
        ha="center",
        fontsize=10
    )

    plt.tight_layout()
    plt.show()


def plot_game_for_player(key, player_filter):
    data = demo_data[key]
    ticks = movement_data[key]["ticks"]
    players = movement_data[key]["players"]
    windows = round_windows_all[key]

    plot_speed_per_round(
        data,
        ticks,
        players,
        windows,
        player_filter=player_filter
    )

# call

# example keys
game_1 = (1, 0)
game_3 = (3, 0)
game_4 = (4, 0)
# example player filters <steamids>
Dottersack = 76561198240903150
Tundra = 76561197987271352
captain_blake = 76561198051680113

plot_game_for_player(game_1, player_filter=captain_blake)
plot_game_for_player(game_3, player_filter=[Dottersack, Tundra])